# 🧪 Create Assay Nodes & Relationships

This notebook reads the dataset manifest, extracts assay and material metadata, maps tissues/cells to UBERON/Cell Ontology, and writes Neo4j CSVs for assay nodes and their relationships to studies, anatomy, cell types, genes, and methylation regions using `genelab_utils` and `ontology_mapper`.

Author: Peter W. Rose, UC San Diego (pwrose.ucsd@gmail.com)

In [1]:
import os
import pandas as pd
import ontology_mapper
import genelab_utils as gl

In [2]:
pd.set_option('display.max_rows', None)  # Shows all rows
pd.set_option('display.max_colwidth', None)  # Shows full content of each cell

## Setup Environment Variables
Edit `../.env` to configure the environment.  

In [3]:
# Node and relationship directory paths
node_dir, rel_dir = gl.setup_environment()

Environment setup for KG version: v0.3.1


In [4]:
# Edit `../.env` to define `BIOPORTAL_API_KEY` for ontology lookups. 
apikey = os.getenv("BIOPORTAL_API_KEY")
if not apikey:
    raise Exception("BIOPORTAL_API_KEY is not set in the .env file!")

In [5]:
MANIFEST_PATH = "../data/manifest.csv"

## Get Info about available Datasets

In [6]:
manifest = pd.read_csv(MANIFEST_PATH, dtype=str, keep_default_na=False,
                       usecols=["identifier", "technology", "measurement", "differential_analysis_method", "assay_name", "organism", "material", "filename", "host_organism", "host_taxonomy", "host_strain"])
# Remove duplicates that have multiple materials. These will be extracted from the factors to ensure a proper mapping
manifest.drop_duplicates(subset=["identifier", "technology", "measurement", "differential_analysis_method", "assay_name", "filename","host_organism", "host_taxonomy", "host_strain"], inplace=True)
manifest.head()

,identifier,technology,measurement,assay_name,material,host_organism,host_strain,filename,differential_analysis_method,organism,host_taxonomy
0,OSD-1,DNA microarray,transcription profiling,OSD-1_transcription-profiling_dna-microarray_Affymetrix,Whole Organism,,,GLDS-1_array_differential_expression.csv,DESeq2,Drosophila melanogaster,
1,OSD-3,DNA microarray,transcription profiling,OSD-3_transcription-profiling_dna-microarray_affymetrix,Whole Organism,,,GLDS-3_array_differential_expression_GLmicroarray.csv,DESeq2,Drosophila melanogaster,
2,OSD-4,DNA microarray,transcription profiling,OSD-4_transcription-profiling_dna-microarray_affymetrix,thymus,,,GLDS-4_array_differential_expression.csv,DESeq2,Mus musculus,
3,OSD-6,DNA microarray,transcription profiling,OSD-6_transcription-profiling_dna-microarray_affymetrix,Cells,,,GLDS-6_array_differential_expression_GLmicroarray.csv,DESeq2,Rattus norvegicus,
4,OSD-13,DNA microarray,transcription profiling,OSD-13_transcription-profiling_dna-microarray_Affymetrix,Primary T Cells,,,GLDS-13_array_differential_expression_GLmicroarray.csv,DESeq2,Homo sapiens,


## Extract Assay Data

In [7]:
variables = {"DESeq2": "Log2fc_",
             "methylKit": "meth.diff_",
             "ANCOM-BC": "Lnfc_",
             "ANCOM-BC2": "Lnfc_",
            }

In [8]:
assays = gl.extract_assay_info(manifest, variables)
# assays.head()
assays[assays["identifier"] == "OSD-267"]

,identifier,technology,measurement,assay_name,material,host_organism,host_strain,filename,differential_analysis_method,organism,host_taxonomy,factors,factors_1,factors_2,factor_space_1,factor_space_2
107,OSD-267,ITS,Amplicon Sequencing,OSD-267_amplicon-sequencing_its_illumina,,Lactuca sativa,Outredgeous,GLDS-267_GAmplicon_ancombc1_differential_abundance_ITS_GLAmpSeq.csv,ANCOM-BC,Microbiota,4236,(Ground Control & root)v(Ground Control & leaf),"[Ground Control, root]","[Ground Control, leaf]",Ground Control,Ground Control
107,OSD-267,ITS,Amplicon Sequencing,OSD-267_amplicon-sequencing_its_illumina,,Lactuca sativa,Outredgeous,GLDS-267_GAmplicon_ancombc1_differential_abundance_ITS_GLAmpSeq.csv,ANCOM-BC,Microbiota,4236,(Space Flight & leaf)v(Ground Control & leaf),"[Space Flight, leaf]","[Ground Control, leaf]",Space Flight,Ground Control
107,OSD-267,ITS,Amplicon Sequencing,OSD-267_amplicon-sequencing_its_illumina,,Lactuca sativa,Outredgeous,GLDS-267_GAmplicon_ancombc1_differential_abundance_ITS_GLAmpSeq.csv,ANCOM-BC,Microbiota,4236,(Space Flight & root)v(Ground Control & leaf),"[Space Flight, root]","[Ground Control, leaf]",Space Flight,Ground Control
107,OSD-267,ITS,Amplicon Sequencing,OSD-267_amplicon-sequencing_its_illumina,,Lactuca sativa,Outredgeous,GLDS-267_GAmplicon_ancombc1_differential_abundance_ITS_GLAmpSeq.csv,ANCOM-BC,Microbiota,4236,(Space Flight & leaf)v(Ground Control & root),"[Space Flight, leaf]","[Ground Control, root]",Space Flight,Ground Control
107,OSD-267,ITS,Amplicon Sequencing,OSD-267_amplicon-sequencing_its_illumina,,Lactuca sativa,Outredgeous,GLDS-267_GAmplicon_ancombc1_differential_abundance_ITS_GLAmpSeq.csv,ANCOM-BC,Microbiota,4236,(Space Flight & root)v(Ground Control & root),"[Space Flight, root]","[Ground Control, root]",Space Flight,Ground Control
107,OSD-267,ITS,Amplicon Sequencing,OSD-267_amplicon-sequencing_its_illumina,,Lactuca sativa,Outredgeous,GLDS-267_GAmplicon_ancombc1_differential_abundance_ITS_GLAmpSeq.csv,ANCOM-BC,Microbiota,4236,(Space Flight & root)v(Space Flight & leaf),"[Space Flight, root]","[Space Flight, leaf]",Space Flight,Space Flight
108,OSD-267,ITS,Amplicon Sequencing,OSD-267_amplicon-sequencing_its_illumina,,Lactuca sativa,Outredgeous,GLDS-267_GAmplicon_deseq2_differential_abundance_ITS_GLAmpSeq.csv,DESeq2,Microbiota,4236,(Ground Control & root)v(Ground Control & leaf),"[Ground Control, root]","[Ground Control, leaf]",Ground Control,Ground Control
108,OSD-267,ITS,Amplicon Sequencing,OSD-267_amplicon-sequencing_its_illumina,,Lactuca sativa,Outredgeous,GLDS-267_GAmplicon_deseq2_differential_abundance_ITS_GLAmpSeq.csv,DESeq2,Microbiota,4236,(Space Flight & leaf)v(Ground Control & leaf),"[Space Flight, leaf]","[Ground Control, leaf]",Space Flight,Ground Control
108,OSD-267,ITS,Amplicon Sequencing,OSD-267_amplicon-sequencing_its_illumina,,Lactuca sativa,Outredgeous,GLDS-267_GAmplicon_deseq2_differential_abundance_ITS_GLAmpSeq.csv,DESeq2,Microbiota,4236,(Space Flight & root)v(Ground Control & leaf),"[Space Flight, root]","[Ground Control, leaf]",Space Flight,Ground Control
108,OSD-267,ITS,Amplicon Sequencing,OSD-267_amplicon-sequencing_its_illumina,,Lactuca sativa,Outredgeous,GLDS-267_GAmplicon_deseq2_differential_abundance_ITS_GLAmpSeq.csv,DESeq2,Microbiota,4236,(Space Flight & leaf)v(Ground Control & root),"[Space Flight, leaf]","[Ground Control, root]",Space Flight,Ground Control


## Extract Material Information and Map to UBERON Ontology

In [9]:
materials = gl.extract_materials(assays)
# Remove rows where 'factors' contains any digit followed by space (e.g., 1 day) since they cannot match any cell or tissue types
materials = materials[~materials["material"].str.contains(r'\d ', na=False)]
materials = materials[materials["material"] != "root"] # plant roots should not be mapped to http://purl.obolibrary.org/obo/NCBITaxon_1
print("Number of materials to map:", materials.shape[0])

Number of materials to map: 442


In [10]:
mapped_materials = ontology_mapper.map_ontology(materials, "material", "material", "UBERON", apikey)
mapped_materials = mapped_materials[mapped_materials["material_id"] != ""].reset_index()

In [11]:
mapped_materials[["material", "material_name", "material_id", "material_uri"]]

,material,material_name,material_id,material_uri
0,3D Cells,cell,CL:0000000,http://purl.obolibrary.org/obo/CL_0000000
1,3D cells,cell,CL:0000000,http://purl.obolibrary.org/obo/CL_0000000
2,Adrenal gland,adrenal gland,UBERON:0002369,http://purl.obolibrary.org/obo/UBERON_0002369
3,Adrenal glands- both sides,adrenal gland,UBERON:0002369,http://purl.obolibrary.org/obo/UBERON_0002369
4,Adult,adult,UBERON:0007023,http://purl.obolibrary.org/obo/UBERON_0007023
5,Blood,blood,UBERON:0000178,http://purl.obolibrary.org/obo/UBERON_0000178
6,Bone Marrow,bone marrow,UBERON:0002371,http://purl.obolibrary.org/obo/UBERON_0002371
7,Brain,brain,UBERON:0000955,http://purl.obolibrary.org/obo/UBERON_0000955
8,Cells,cell,CL:0000000,http://purl.obolibrary.org/obo/CL_0000000
9,Cerebellum,cerebellum,UBERON:0002037,http://purl.obolibrary.org/obo/UBERON_0002037


## Create Anatomy (proxy) Nodes

In [12]:
material_ids = mapped_materials[["material_id"]].copy()
material_ids.rename(columns={"material_id": "identifier"}, inplace=True)
anatomy = material_ids[material_ids["identifier"].str.startswith("UBERON:")].copy()

In [13]:
anatomy_nodes = gl.save_dataframe_to_kg(anatomy, 'Anatomy', node_dir)
print(f"Number of Anatomy nodes: {anatomy_nodes.shape[0]}")
anatomy_nodes.head()

Number of Anatomy nodes: 42


,identifier
2,UBERON:0002369
4,UBERON:0007023
5,UBERON:0000178
6,UBERON:0002371
7,UBERON:0000955


## Create CellType (proxy) Nodes

In [14]:
cell_type = material_ids[material_ids["identifier"].str.startswith("CL:")].copy()

In [15]:
cell_type_nodes = gl.save_dataframe_to_kg(cell_type, 'CellType', node_dir)
print(f"Number of CellType nodes: {cell_type_nodes.shape[0]}")
cell_type_nodes.head()

Number of CellType nodes: 7


,identifier
0,CL:0000000
16,CL:4030029
29,CL:0000056
30,CL:0000081
31,CL:0000084


## Assign Materials for Assay Factors 1 and 2

In [16]:
assays = gl.assign_material_to_assays(assays, mapped_materials)

## Assign Unique Assay Identifier

In [17]:
assays = gl.add_assay_identifiers(assays)

## Create Study-PERFORMED_SpAS-Assay Relationships

In [18]:
study_performed_assay = assays[["study_id", "identifier"]].copy()
study_performed_assay.rename(columns={"study_id": "from", "identifier": "to"}, inplace=True)

In [19]:
study_performed_assay_rels = gl.save_dataframe_to_kg(study_performed_assay, 'Study-PERFORMED_SpAS-Assay', rel_dir)
print(f"Number of Study-PERFORMED_SpAS-Assay relationships: {study_performed_assay_rels.shape[0]}")
study_performed_assay_rels.head()

Number of Study-PERFORMED_SpAS-Assay relationships: 13146


,from,to
0,OSD-1,OSD-1-0a8f911676573a415c92f4b393ca6494
0,OSD-1,OSD-1-8b2101b14c2055023c5bc52e6742c2f0
0,OSD-1,OSD-1-93764f2c45bd619d5201149ef8ad40ca
0,OSD-1,OSD-1-a6cf1b5426cf6cfbb50574cbc18dbc89
0,OSD-1,OSD-1-341cb38e550f2b1d5b59b6b01c4b183f


## Create Assay-INVESTIGATED_ASiA-Anatomy Relationships

In [20]:
assay_investigated_material_1 = assays[["identifier", "material_id_1"]].copy()
assay_investigated_material_1.rename(columns={"identifier": "from", "material_id_1": "to"}, inplace=True)

assay_investigated_material_2 = assays[["identifier", "material_id_2"]].copy()
assay_investigated_material_2.rename(columns={"identifier": "from", "material_id_2": "to"}, inplace=True)

assay_investigated_material = pd.concat([assay_investigated_material_1, assay_investigated_material_2]).drop_duplicates()

assay_investigated_anatomy = assay_investigated_material[assay_investigated_material["to"].str.startswith("UBERON:")]

In [21]:
assay_investigated_anatomy_rels = gl.save_dataframe_to_kg(assay_investigated_anatomy, 'Assay-INVESTIGATED_ASiA-Anatomy', rel_dir)
print(f"Number of Assay-INVESTIGATED_ASiA-Anatomy relationships: {assay_investigated_anatomy_rels.shape[0]}")
assay_investigated_anatomy_rels.head()

Number of Assay-INVESTIGATED_ASiA-Anatomy relationships: 7214


,from,to
0,OSD-1-0a8f911676573a415c92f4b393ca6494,UBERON:0000468
0,OSD-1-8b2101b14c2055023c5bc52e6742c2f0,UBERON:0000468
0,OSD-1-93764f2c45bd619d5201149ef8ad40ca,UBERON:0000468
0,OSD-1-a6cf1b5426cf6cfbb50574cbc18dbc89,UBERON:0000468
0,OSD-1-341cb38e550f2b1d5b59b6b01c4b183f,UBERON:0000468


## Create Assay-INVESTIGATED_ASiCT-CellType Relationships

In [22]:
assay_investigated_cell_type = assay_investigated_material[assay_investigated_material["to"].str.startswith("CL:")]

In [23]:
assay_investigated_cell_type_rels = gl.save_dataframe_to_kg(assay_investigated_cell_type, 'Assay-INVESTIGATED_ASiCT-CellType', rel_dir)
print(f"Number of Assay-INVESTIGATED_ASiCT-CellType: {assay_investigated_cell_type_rels.shape[0]}")
assay_investigated_cell_type_rels.head()

Number of Assay-INVESTIGATED_ASiCT-CellType: 1240


,from,to
3,OSD-6-9068ec75d9792d5b76bf305502dbb66c,CL:0000000
3,OSD-6-b2baf006c2865ceafde3240084021f03,CL:0000000
4,OSD-13-7a4f1e932d99e95949fcffe85bf19721,CL:0000084
4,OSD-13-f66bf24297612cd7012daa107b820ad2,CL:0000084
4,OSD-13-5dba97ded298e763b6d727124f3c3f79,CL:0000084


## Create Assay Nodes

In [24]:
assays.rename(columns={'assay_name': 'name'}, inplace=True)
assay_props = assays[["identifier", "name", "technology", 
                      "measurement", "differential_analysis_method",
                      "factor_space_1", "factor_space_2",
                      "factors_1", "factors_2", 
                      "material_1", "material_2", 
                      "material_name_1", "material_name_2",
                      "material_id_1", "material_id_2",
                     ]].copy()

assay_nodes = gl.save_dataframe_to_kg(assay_props, 'Assay', node_dir)
print(f"Number of Assays nodes: {assay_nodes.shape[0]}")
assay_nodes.head()

Number of Assays nodes: 13146


,identifier,name,technology,measurement,differential_analysis_method,factor_space_1,factor_space_2,factors_1,factors_2,material_1,material_2,material_name_1,material_name_2,material_id_1,material_id_2
0,OSD-1-0a8f911676573a415c92f4b393ca6494,OSD-1_transcription-profiling_dna-microarray_Affymetrix,DNA microarray,transcription profiling,DESeq2,Ground Control,Ground Control,Ground Control|infected with Beauveria bassiana,Ground Control|infected with Escherichia coli,Whole Organism,Whole Organism,whole organism,whole organism,UBERON:0000468,UBERON:0000468
0,OSD-1-8b2101b14c2055023c5bc52e6742c2f0,OSD-1_transcription-profiling_dna-microarray_Affymetrix,DNA microarray,transcription profiling,DESeq2,Ground Control,Ground Control,Ground Control|infected with Beauveria bassiana,Ground Control|uninfected,Whole Organism,Whole Organism,whole organism,whole organism,UBERON:0000468,UBERON:0000468
0,OSD-1-93764f2c45bd619d5201149ef8ad40ca,OSD-1_transcription-profiling_dna-microarray_Affymetrix,DNA microarray,transcription profiling,DESeq2,Ground Control,Space Flight,Ground Control|infected with Beauveria bassiana,Space Flight|infected with Beauveria bassiana,Whole Organism,Whole Organism,whole organism,whole organism,UBERON:0000468,UBERON:0000468
0,OSD-1-a6cf1b5426cf6cfbb50574cbc18dbc89,OSD-1_transcription-profiling_dna-microarray_Affymetrix,DNA microarray,transcription profiling,DESeq2,Ground Control,Space Flight,Ground Control|infected with Beauveria bassiana,Space Flight|infected with Escherichia coli,Whole Organism,Whole Organism,whole organism,whole organism,UBERON:0000468,UBERON:0000468
0,OSD-1-341cb38e550f2b1d5b59b6b01c4b183f,OSD-1_transcription-profiling_dna-microarray_Affymetrix,DNA microarray,transcription profiling,DESeq2,Ground Control,Space Flight,Ground Control|infected with Beauveria bassiana,Space Flight|uninfected,Whole Organism,Whole Organism,whole organism,whole organism,UBERON:0000468,UBERON:0000468


In [25]:
assay_measured_mgene = gl.extract_transcription_data(assays, threshold=0.1)

extract_transcription_data: 0.1
processing: OSD-100
processing: OSD-101
processing: OSD-102
processing: OSD-103
processing: OSD-104
processing: OSD-105
processing: OSD-109
No statistically significant data for OSD-109: Log2fc_(Fe-56 ion radiation & 1 day)v(sham-irradiated & 1 day)
No statistically significant data for OSD-109: Log2fc_(sham-irradiated & 1 day)v(Fe-56 ion radiation & 1 day)
processing: OSD-112
processing: OSD-113
processing: OSD-117
No statistically significant data for OSD-117: Log2fc_(proton & 1 day)v(proton & 12 day)
No statistically significant data for OSD-117: Log2fc_(proton & 1 day)v(proton & 5 day)
No statistically significant data for OSD-117: Log2fc_(proton & 1 day)v(sham-irradiated & 1 day)
No statistically significant data for OSD-117: Log2fc_(proton & 1 day)v(sham-irradiated & 3 day)
No statistically significant data for OSD-117: Log2fc_(proton & 12 day)v(proton & 26 day)
No statistically significant data for OSD-117: Log2fc_(proton & 12 day)v(proton & 5 day

## Create Assay-MEASURED_DIFFERENTIAL_EXPRESSION_ASmMG-MGene Relationships

In [26]:
assay_measured_mgene_rels = gl.save_dataframe_to_kg(assay_measured_mgene, 'Assay-MEASURED_DIFFERENTIAL_EXPRESSION_ASmMG-MGene', rel_dir)
print(f"Number of Assay-MEASURED_DIFFERENTIAL_EXPRESSION_ASmMG-MGene relationships: {assay_measured_mgene_rels.shape[0]}")
assay_measured_mgene_rels.head()

Number of Assay-MEASURED_DIFFERENTIAL_EXPRESSION_ASmMG-MGene relationships: 50208066


,from,to,log2fc,adj_p_value,group_mean_1,group_stdev_1,group_mean_2,group_stdev_2
0,OSD-100-b0da37dd831c8f592bf554cbc4516cb6,23849,0.211489,0.001988,3888.1400,165.145,3358.249,145.5544
1,OSD-100-b0da37dd831c8f592bf554cbc4516cb6,235339,0.355894,0.007801,2915.2860,319.337,2278.336,200.6251
2,OSD-100-b0da37dd831c8f592bf554cbc4516cb6,12444,0.300122,0.040228,7583.9740,801.5491,6159.397,615.0515
3,OSD-100-b0da37dd831c8f592bf554cbc4516cb6,66108,0.293361,0.023020,2644.8020,267.1048,2158.644,161.285
4,OSD-100-b0da37dd831c8f592bf554cbc4516cb6,14395,-0.430303,0.074501,228.5563,40.01388,307.851,37.46778


In [27]:
methylation_data = gl.extract_methylation_data(assays, threshold=0.1)

extract_methylation_data: 0.1
processing: OSD-103
processing: OSD-103
processing: OSD-105
processing: OSD-105
processing: OSD-47
No significant data for OSD-47: meth.diff_(Ground Control)v(Space Flight)
No significant data for OSD-47: meth.diff_(Space Flight)v(Ground Control)
processing: OSD-47
processing: OSD-48
processing: OSD-48


## Create MethylationRegion Nodes

In [28]:
methylation_data["name"] = methylation_data["methylation_id"]
methylation_region = methylation_data[["methylation_id", "name", "chr", "start", "end", "dist.to.feature", "in_promoter", "in_exon", "in_intron"]].copy()
methylation_region.rename(columns={"methylation_id": "identifier", "chr": "chromosome", "dist.to.feature": "dist_to_feature"}, inplace=True)
methylation_region["dist_to_feature"] = methylation_region["dist_to_feature"].astype(int)

In [29]:
methylation_region_nodes = gl.save_dataframe_to_kg(methylation_region, 'MethylationRegion', node_dir)
print(f"Number of MethylationRegion nodes: {methylation_region_nodes.shape[0]}")
methylation_region_nodes.head()

Number of MethylationRegion nodes: 8003


,identifier,name,chromosome,start,end,dist_to_feature,in_promoter,in_exon,in_intron
0,11:58964417-58964417,11:58964417-58964417,11,58964417,58964417,6102,false,true,false
1,11:67149779-67149779,11:67149779-67149779,11,67149779,67149779,-18171,false,true,false
2,11:67149789-67149789,11:67149789-67149789,11,67149789,67149789,-18161,false,true,false
3,13:45023086-45023086,13:45023086-45023086,13,45023086,45023086,28829,false,false,true
4,13:45023089-45023089,13:45023089-45023089,13,45023089,45023089,28832,false,false,true


## Create Assay-MEASURED_DIFFERENTIAL_METHYLATION_ASmMR-MethylationRegion Relationships

In [30]:
assay_measured_methylation_region = methylation_data[["assay_id", "methylation_id", "methylation_diff", "q_value", "group_mean_1", "group_stdev_1", "group_mean_2", "group_stdev_2"]].copy()
assay_measured_methylation_region.rename(columns={"assay_id": "from", "methylation_id": "to"}, inplace=True)

In [31]:
assay_measured_methylation_region_rel = gl.save_dataframe_to_kg(assay_measured_methylation_region, 'Assay-MEASURED_DIFFERENTIAL_METHYLATION_ASmMR-MethylationRegion', rel_dir)
print(f"Number of Assay-MEASURED_DIFFERENTIAL_METHYLATION_ASmMR-MethylationRegion relationships: {assay_measured_methylation_region_rel.shape[0]}")
assay_measured_methylation_region_rel.head()

Number of Assay-MEASURED_DIFFERENTIAL_METHYLATION_ASmMR-MethylationRegion relationships: 27454


,from,to,methylation_diff,q_value,group_mean_1,group_stdev_1,group_mean_2,group_stdev_2
0,OSD-103-b2dd97caa6abbb454a11e76982976b20,11:58964417-58964417,76.875000,1.050214e-02,96.666667,5.773503,20.0,
1,OSD-103-b2dd97caa6abbb454a11e76982976b20,11:67149779-67149779,-19.736842,7.069545e-02,8.676471,12.923618,29.115135,17.95668
2,OSD-103-b2dd97caa6abbb454a11e76982976b20,11:67149789-67149789,-30.701754,2.195020e-02,11.445052,14.519582,43.669591,22.609147
3,OSD-103-b2dd97caa6abbb454a11e76982976b20,13:45023086-45023086,6.768814,1.477567e-02,12.107623,,5.338809,
4,OSD-103-b2dd97caa6abbb454a11e76982976b20,13:45023089-45023089,9.429868,1.067579e-08,14.615385,,5.402405,2.036044


## Create MGene-METHYLATED_IN_MGmMR-MethylationRegion Relationships

In [32]:
mgene_methylated_in_methylation_region = methylation_data[["ENTREZID", "methylation_id"]].copy()
mgene_methylated_in_methylation_region.rename(columns={"ENTREZID": "from", "methylation_id": "to"}, inplace=True)

In [33]:
mgene_methylated_in_methylation_region_rels = gl.save_dataframe_to_kg(mgene_methylated_in_methylation_region, 'MGene-METHYLATED_IN_MGmMR-MethylationRegion', rel_dir)
print(f"Number of MGene-METHYLATED_IN_MGmMR-MethylationRegion relationships: {mgene_methylated_in_methylation_region_rels.shape[0]}")
mgene_methylated_in_methylation_region_rels.head()

Number of MGene-METHYLATED_IN_MGmMR-MethylationRegion relationships: 8270


,from,to
0,380698,11:58964417-58964417
1,17885,11:67149779-67149779
2,17885,11:67149789-67149789
3,16468,13:45023086-45023086
4,16468,13:45023089-45023089


## Create Assay-MEASURED_DIFFERENTIAL_ABUNDANCE_ASmO-Organism Relationships

In [34]:
amplicon_deseq2_data = gl.extract_amplicon_deseq2_data(assays, threshold=0.1)
amplicon_deseq2_data.head()

extract_amplicon_data: 0.1
processing: OSD-267
No statistically significant data for OSD-267: Log2fc_(Ground Control & root)v(Ground Control & leaf)
No statistically significant data for OSD-267: Log2fc_(Space Flight & leaf)v(Ground Control & leaf)
No statistically significant data for OSD-267: Log2fc_(Space Flight & leaf)v(Ground Control & root)
processing: OSD-267
No statistically significant data for OSD-267: Log2fc_(Ground Control & root)v(Ground Control & leaf)
No statistically significant data for OSD-267: Log2fc_(Space Flight & leaf)v(Ground Control & leaf)
No statistically significant data for OSD-267: Log2fc_(Space Flight & leaf)v(Ground Control & root)


,from,to,name,log2fc,adj_p_value,group_mean_1,group_stdev_1,group_mean_2,group_stdev_2
0,OSD-267-3a5b715be98f5f44358815ce51be8cfb,48736,Ralstonia,9.636646,0.002716,12559.719050,1194.304354,15.738135,22.131337
1,OSD-267-3a5b715be98f5f44358815ce51be8cfb,1236,Gammaproteobacteria,10.648586,0.002716,13365.870841,12006.217538,8.303712,12.674965
2,OSD-267-3a5b715be98f5f44358815ce51be8cfb,563835,Chitinophagaceae,10.602168,0.030632,5828.540324,2908.677862,3.734301,7.468601
3,OSD-267-3a5b715be98f5f44358815ce51be8cfb,149698,Massilia,14.277971,0.000138,6235.835088,4556.988484,0.222610,0.445220
4,OSD-267-3a5b715be98f5f44358815ce51be8cfb,335928,Xanthobacteraceae,9.939449,0.043273,4700.277381,3076.509326,4.771606,9.543213


In [35]:
assay_measured_organism_deseq2 = amplicon_deseq2_data[["from", "to", "log2fc", "adj_p_value", "group_mean_1", "group_stdev_1", "group_mean_2", "group_stdev_2"]].copy()
assay_measured_organism_deseq2["lnfc"] = ""
assay_measured_organism_deseq2["q_value"] = ""
assay_measured_organism_deseq2 = assay_measured_organism_deseq2[["from", "to", "log2fc", "adj_p_value", "lnfc", "q_value", "group_mean_1", "group_stdev_1", "group_mean_2", "group_stdev_2"]]

In [36]:
amplicon_ancombbc_data = gl.extract_amplicon_ancombc_data(assays, threshold=0.1)
amplicon_ancombbc_data.head()

extract_amplicon_data: 0.1
ANCOM-BC: GLDS-267_GAmplicon_ancombc1_differential_abundance_16S_GLAmpSeq.csv
processing: OSD-267
ANCOM-BC: GLDS-267_GAmplicon_ancombc1_differential_abundance_ITS_GLAmpSeq.csv
processing: OSD-267
No statistically significant data for OSD-267: Lnfc_(Ground Control & root)v(Ground Control & leaf)
No statistically significant data for OSD-267: Lnfc_(Space Flight & leaf)v(Ground Control & leaf)
No statistically significant data for OSD-267: Lnfc_(Space Flight & leaf)v(Ground Control & root)


,from,to,name,lnfc,log2fc,q_value,group_mean_1,group_stdev_1,group_mean_2,group_stdev_2
0,OSD-267-7ed6de00a8856e7dd9fd3448b9237efc,149698,Massilia,2.113977,3.049824,3.110471e-54,2.249905,0.05268,0.138629,0.277259
1,OSD-267-7ed6de00a8856e7dd9fd3448b9237efc,2,Bacteria,-4.127140,-5.954205,1.110451e-113,0.000000,0.00000,4.129842,0.430230
2,OSD-267-7ed6de00a8856e7dd9fd3448b9237efc,2,Bacteria,-3.545169,-5.114598,2.983644e-77,0.000000,0.00000,3.547871,0.471363
3,OSD-267-7ed6de00a8856e7dd9fd3448b9237efc,469,Acinetobacter,-2.540478,-3.665135,1.405283e-04,0.000000,0.00000,2.543180,1.389963
4,OSD-267-7ed6de00a8856e7dd9fd3448b9237efc,2762318,Weeksellaceae,-1.793282,-2.587160,5.076568e-02,0.000000,0.00000,1.795984,1.467392


In [37]:
assay_measured_organism_ancombbc = amplicon_ancombbc_data[["from", "to", "log2fc", "lnfc", "q_value", "group_mean_1", "group_stdev_1", "group_mean_2", "group_stdev_2"]].copy()
assay_measured_organism_ancombbc["adj_p_value"] = ""
assay_measured_organism_ancombbc = assay_measured_organism_ancombbc[["from", "to", "log2fc", "adj_p_value", "lnfc", "q_value", "group_mean_1", "group_stdev_1", "group_mean_2", "group_stdev_2"]]

In [38]:
assay_measured_organism = pd.concat([assay_measured_organism_deseq2, assay_measured_organism_ancombbc])

In [39]:
assay_measured_organism_rels = gl.save_dataframe_to_kg(assay_measured_organism, 'Assay-MEASURED_DIFFERENTIAL_ABUNDANCE_ASmO-Organism', rel_dir)
print(f"Number of MEASURED_DIFFERENTIAL_ABUNDANCE relationships: {assay_measured_organism_rels.shape[0]}")
assay_measured_organism_rels.head()

Number of MEASURED_DIFFERENTIAL_ABUNDANCE relationships: 193


,from,to,log2fc,adj_p_value,lnfc,q_value,group_mean_1,group_stdev_1,group_mean_2,group_stdev_2
0,OSD-267-3a5b715be98f5f44358815ce51be8cfb,48736,9.636646,0.002716,,,12559.719050,1194.304354,15.738135,22.131337
1,OSD-267-3a5b715be98f5f44358815ce51be8cfb,1236,10.648586,0.002716,,,13365.870841,12006.217538,8.303712,12.674965
2,OSD-267-3a5b715be98f5f44358815ce51be8cfb,563835,10.602168,0.030632,,,5828.540324,2908.677862,3.734301,7.468601
3,OSD-267-3a5b715be98f5f44358815ce51be8cfb,149698,14.277971,0.000138,,,6235.835088,4556.988484,0.222610,0.445220
4,OSD-267-3a5b715be98f5f44358815ce51be8cfb,335928,9.939449,0.043273,,,4700.277381,3076.509326,4.771606,9.543213


In [40]:
assay_measured_organism_rels.tail()

,from,to,log2fc,adj_p_value,lnfc,q_value,group_mean_1,group_stdev_1,group_mean_2,group_stdev_2
188,OSD-267-1b615db99ea709f26107c3c34c1e5280,1755443,9.551005,,6.620252,0.0,8.661617,0.316161,0.000000,0.000000
189,OSD-267-1b615db99ea709f26107c3c34c1e5280,2877796,7.563685,,5.242747,0.0,7.284111,0.183260,0.000000,0.000000
190,OSD-267-1b615db99ea709f26107c3c34c1e5280,1756125,-7.710757,,-5.34469,0.032824,0.000000,0.000000,3.303325,3.303325
191,OSD-267-8d3e446e1dcb0c102e9b5095bf5be1a9,1755443,8.762779,,6.073895,0.016308,8.661617,0.316161,1.907862,2.698124
192,OSD-267-8d3e446e1dcb0c102e9b5095bf5be1a9,2877796,9.527922,,6.604252,0.0,7.284111,0.183260,0.000000,0.000000


## Create Organism Nodes

In [41]:
organisms = pd.concat([amplicon_deseq2_data[["to", "name"]], amplicon_ancombbc_data[["to", "name"]]]).copy()

In [42]:
organisms.drop_duplicates(inplace=True)
organisms.sort_values(by="to", inplace=True)
organisms.rename(columns={"to": "identifier"}, inplace=True)

In [43]:
organism_nodes = gl.save_dataframe_to_kg(organisms, 'Organism', node_dir)
print(f"Number of Organism nodes: {organism_nodes.shape[0]}")
organism_nodes.head()

Number of Organism nodes: 46


,identifier,name
1,2,Bacteria
11,81,Hyphomicrobium
9,286,Pseudomonas
10,374,Bradyrhizobium
9,407,Methylobacterium
